In [1]:
#Loading .xlsx file in python 
import pandas as pd
df = pd.read_excel("customer_journey.xlsx")
df.head()

,Session_ID,User_ID,Time_stamp,Page_Type,Device_Type,Country,Referral_Source,Time_On_Page(seconds),Items_In_Cart,Purchased
0,session_0,user_2223,2025-01-20 22:53:34,home,Desktop,India,Social Media,55,0,0
1,session_1,user_2192,2025-02-26 12:57:10,home,Tablet,Germany,Email,99,0,0
2,session_1,user_2192,2025-02-26 12:59:11,product,Tablet,Germany,Email,121,0,0
3,session_2,user_1708,2025-06-24 15:40:46,home,Mobile,India,Google,160,0,0
4,session_3,user_2976,2025-06-11 07:21:02,home,Tablet,UK,Google,113,0,0


In [2]:
# Data vailation
df.info()
df.isnull().sum()
df.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12719 entries, 0 to 12718
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Session_ID             12719 non-null  object        
 1   User_ID                12719 non-null  object        
 2   Time_stamp             12719 non-null  datetime64[ns]
 3   Page_Type              12719 non-null  object        
 4   Device_Type            12719 non-null  object        
 5   Country                12719 non-null  object        
 6   Referral_Source        12719 non-null  object        
 7   Time_On_Page(seconds)  12719 non-null  int64         
 8   Items_In_Cart          12719 non-null  int64         
 9   Purchased              12719 non-null  int64         
dtypes: datetime64[ns](1), int64(3), object(6)
memory usage: 993.8+ KB


np.int64(0)

In [3]:
# Funnel flags creation
df['product_view'] = (df['Page_Type'] == 'product').astype(int)
df['add_to_cart'] = (df['Items_In_Cart'] > 0).astype(int)
df['purchase_flag'] = df['Purchased']
df[['Page_Type', 'product_view']].head(10)

,Page_Type,product_view
0,home,0
1,home,0
2,product,1
3,home,0
4,home,0
5,home,0
6,product,1
7,home,0
8,product,1
9,cart,0


In [4]:
# User-level aggregation
user_df = df.groupby('User_ID').agg({
    'product_view': 'max',
    'add_to_cart': 'max',
    'purchase_flag': 'max'
}).reset_index()
print(user_df.head())

     User_ID  product_view  add_to_cart  purchase_flag
0  user_1001             1            1              1
1  user_1002             1            1              1
2  user_1004             0            0              0
3  user_1005             1            1              1
4  user_1006             1            1              0


In [5]:
# Funnel numbers
total_users = user_df.shape[0]
view_users = user_df['product_view'].sum()
cart_users = user_df['add_to_cart'].sum()
purchase_users = user_df['purchase_flag'].sum()

In [6]:
# Funnel Table
funnel = pd.DataFrame({
    'Stage': ['Visit', 'Product View', 'Add to Cart', 'Purchase'],
    'Users': [total_users, view_users, cart_users, purchase_users]
})
funnel['Conversion Rate'] = funnel['Users'] / funnel['Users'].shift(1)
print(funnel)

          Stage  Users  Conversion Rate
0         Visit   1872              NaN
1  Product View   1763         0.941774
2   Add to Cart   1548         0.878049
3      Purchase    792         0.511628


In [7]:
# Segementation Analysis
user_device = df.groupby('User_ID')['Device_Type'].first().reset_index()
user_df = user_df.merge(user_device, on='User_ID', how='left')
device_funnel = user_df.groupby('Device_Type').agg({
    'User_ID': 'count',
    'product_view': 'sum',
    'add_to_cart': 'sum',
    'purchase_flag': 'sum'
}).reset_index()
print(device_funnel)

  Device_Type  User_ID  product_view  add_to_cart  purchase_flag
0     Desktop      654           603          520            277
1      Mobile      588           561          501            256
2      Tablet      630           599          527            259


In [8]:
# Conversion Rates
device_funnel['view_rate'] = device_funnel['product_view'] / device_funnel['User_ID']
device_funnel['cart_rate'] = device_funnel['add_to_cart'] / device_funnel['product_view']
device_funnel['purchase_rate'] = device_funnel['purchase_flag'] / device_funnel['add_to_cart']
print(device_funnel)

  Device_Type  User_ID  product_view  add_to_cart  purchase_flag  view_rate  \
0     Desktop      654           603          520            277   0.922018   
1      Mobile      588           561          501            256   0.954082   
2      Tablet      630           599          527            259   0.950794   

   cart_rate  purchase_rate  
0   0.862355       0.532692  
1   0.893048       0.510978  
2   0.879800       0.491461  


In [9]:
# Referral Source Analysis
user_referral = df.groupby('User_ID')['Referral_Source'].first().reset_index()
user_df = user_df.merge(user_referral, on='User_ID', how='left')
referral_funnel = user_df.groupby('Referral_Source').agg({
    'User_ID': 'count',
    'product_view': 'sum',
    'add_to_cart': 'sum',
    'purchase_flag': 'sum'
}).reset_index()
print(referral_funnel)

  Referral_Source  User_ID  product_view  add_to_cart  purchase_flag
0          Direct      459           423          359            196
1           Email      467           441          388            193
2          Google      476           453          401            207
3    Social Media      470           446          400            196
